In [9]:
import pandas as pd 

df = pd.read_csv(
   "ml-100k/u.data",
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"],
)
print(df.shape)
print(df["user_id"].nunique(), df["item_id"].nunique())

df = df.sort_values(["user_id","timestamp"])

df["rank_latest"] = df.groupby("user_id")["timestamp"].rank(method="first" , ascending=False)

test = df[df["rank_latest"] == 1]
val = df[df["rank_latest"] == 2]
train = df[df["rank_latest"]>2]

print(len(test),len(val),len(train))


(100000, 4)
943 1682
943 943 98114


In [15]:
import numpy as np

ranked_items = train["item_id"].value_counts().index.to_numpy()
seen = pd.concat([train, val]).groupby("user_id")["item_id"].apply(set).to_dict()

def popularity_ranker(user_id):
  return [i for i in ranked_items if i not in seen[user_id] ]

def evaluate(ranker, eval_df=test, K=10):
    hits, ndcgs = [], []
    for row in eval_df.itertuples():
        topk = ranker(row.user_id)[:K]
        if row.item_id in topk:
            pos = topk.index(row.item_id)          
            hits.append(1)
            ndcgs.append(1 / np.log2(pos + 2))     
        else:
            hits.append(0)
            ndcgs.append(0)
    return np.mean(hits), np.mean(ndcgs)

recall10, ndcg10 = evaluate(popularity_ranker)
print(f"Popularity  Recall@10: {recall10:.4f}   NDCG@10: {ndcg10:.4f}")

Popularity  Recall@10: 0.0859   NDCG@10: 0.0426


In [11]:
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

n_users , n_items = 943 , 1682

seen_train = train.groupby("user_id")["item_id"].apply(set).to_dict()
seen_trainval = seen

def build_matrix(data, binary=True):
    vals = np.ones(len(data)) if binary else data["rating"].to_numpy(dtype=float)
    return csr_matrix(
        (vals, (data["user_id"].to_numpy() - 1, data["item_id"].to_numpy() - 1)),
        shape=(n_users, n_items),
    )
def svd_scores(M, k):
    svd = TruncatedSVD(n_components=k, random_state=0)

    user_factors = svd.fit_transform(M)
    item_factors = svd.components_

    return user_factors @ item_factors
def make_ranker(scores, seen):
    def ranker(user_id):
        s = scores[user_id - 1].copy()

        s[[i - 1 for i in seen[user_id]]] = -np.inf

        return (np.argsort(-s)[:50] + 1).tolist()

    return ranker
pop_val = lambda u: [i for i in ranked_items if i not in seen_train[u]]

pop_r, pop_n = evaluate(pop_val, eval_df=val)
M_train = build_matrix(train)

rows = [("Popularity", "-", pop_r, pop_n)]

for k in [8, 16, 32, 64, 128]:
    scores = svd_scores(M_train, k)
    r, n = evaluate(make_ranker(scores, seen_train), eval_df=val)
    rows.append(("SVD", k, r, n))

results = pd.DataFrame(
    rows,
    columns=["model", "k", "Recall@10", "NDCG@10"]
)

print(results.to_string(index=False))


     model   k  Recall@10  NDCG@10
Popularity   -   0.075292 0.037655
       SVD   8   0.131495 0.068483
       SVD  16   0.151644 0.080505
       SVD  32   0.159067 0.080894
       SVD  64   0.160127 0.077461
       SVD 128   0.142100 0.076457


In [12]:
best_k = int(results[results["model"] == "SVD"].sort_values("NDCG@10", ascending=False).iloc[0]["k"])
print("Best k on val:", best_k)

M_full = build_matrix(pd.concat([train, val]))
scores = svd_scores(M_full, best_k)
r, n = evaluate(make_ranker(scores, seen_trainval), eval_df=test)
print(f"SVD (k={best_k})  Recall@10: {r:.4f}   NDCG@10: {n:.4f}")

Best k on val: 32
SVD (k=32)  Recall@10: 0.1389   NDCG@10: 0.0728


In [16]:
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)

class NCF(nn.Module):
    def __init__(self, n_users, n_items, dim=32, dropout=0.2):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_items, dim)
        nn.init.normal_(self.user_emb.weight, std=0.05)
        nn.init.normal_(self.item_emb.weight, std=0.05)
        self.mlp = nn.Sequential(
            nn.Linear(2 * dim, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32),      nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, u, i):
        x = torch.cat([self.user_emb(u), self.item_emb(i)], dim=1)   
        return self.mlp(x).squeeze(1)                               

def sample_negatives(users, pos_mat, n_neg=4):
    """For every positive, draw n_neg random items the user has NOT interacted with."""
    users_rep = np.repeat(users, n_neg)
    neg = np.random.randint(0, n_items, size=len(users_rep))
    clash = pos_mat[users_rep, neg]                  
    while clash.any():
        neg[clash] = np.random.randint(0, n_items, size=clash.sum())
        clash = pos_mat[users_rep, neg]
    return users_rep, neg

@torch.no_grad()
def ncf_scores(model):
    """Score every (user, item) pair -> array of shape (n_users, n_items)."""
    model.eval()
    u = torch.arange(n_users).repeat_interleave(n_items)
    i = torch.arange(n_items).repeat(n_users)
    chunks = [model(u[s:s + 100_000], i[s:s + 100_000]) for s in range(0, len(u), 100_000)]
    return torch.cat(chunks).view(n_users, n_items).numpy()

def train_ncf(data, dim=32, dropout=0.2, wd=1e-5, epochs=15, lr=1e-3,
              eval_df=None, eval_seen=None):
    pos_mat = build_matrix(data).toarray().astype(bool)
    users = data["user_id"].to_numpy() - 1
    items = data["item_id"].to_numpy() - 1

    model = NCF(n_users, n_items, dim, dropout)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    loss_fn = nn.BCEWithLogitsLoss()
    history = []

    for ep in range(1, epochs + 1):
        model.train()
        neg_u, neg_i = sample_negatives(users, pos_mat)         
        U = np.concatenate([users, neg_u])
        I = np.concatenate([items, neg_i])
        y = np.concatenate([np.ones(len(users)), np.zeros(len(neg_u))])
        perm = np.random.permutation(len(U))
        U = torch.tensor(U[perm], dtype=torch.long)
        I = torch.tensor(I[perm], dtype=torch.long)
        y = torch.tensor(y[perm], dtype=torch.float32)

        for s in range(0, len(U), 1024):                        
            u, i, yb = U[s:s + 1024], I[s:s + 1024], y[s:s + 1024]
            opt.zero_grad()
            loss = loss_fn(model(u, i), yb)
            loss.backward()
            opt.step()

        if eval_df is not None:
            r, n = evaluate(make_ranker(ncf_scores(model), eval_seen), eval_df=eval_df)
            history.append((ep, r, n))
    return model, history

In [17]:
ncf_rows = []
for dim in [16, 32]:
    for dropout in [0.1, 0.3]:
        _, hist = train_ncf(train, dim=dim, dropout=dropout, wd=1e-5,
                            eval_df=val, eval_seen=seen_train)
        for ep, r, n in hist:
            ncf_rows.append((dim, dropout, ep, r, n))
        best = max(hist, key=lambda h: h[2])
        print(f"dim={dim} dropout={dropout}  best epoch={best[0]}  "
              f"Recall@10={best[1]:.4f}  NDCG@10={best[2]:.4f}")

ncf_results = pd.DataFrame(ncf_rows, columns=["dim", "dropout", "epoch", "Recall@10", "NDCG@10"])
best_cfg = ncf_results.sort_values("NDCG@10", ascending=False).iloc[0]
print("\nBest config on val:\n", best_cfg)

dim=16 dropout=0.1  best epoch=13  Recall@10=0.1156  NDCG@10=0.0606
dim=16 dropout=0.3  best epoch=13  Recall@10=0.1145  NDCG@10=0.0573
dim=32 dropout=0.1  best epoch=12  Recall@10=0.1315  NDCG@10=0.0683
dim=32 dropout=0.3  best epoch=15  Recall@10=0.1220  NDCG@10=0.0642

Best config on val:
 dim          32.000000
dropout       0.100000
epoch        12.000000
Recall@10     0.131495
NDCG@10       0.068252
Name: 41, dtype: float64


In [19]:
model, _ = train_ncf(pd.concat([train, val]),
                     dim=int(best_cfg["dim"]), dropout=float(best_cfg["dropout"]),
                     wd=1e-5, epochs=int(best_cfg["epoch"]))

r, n = evaluate(make_ranker(ncf_scores(model), seen_trainval), eval_df=test)
print(f"NCF  Recall@10: {r:.4f}   NDCG@10: {n:.4f}")

NCF  Recall@10: 0.1177   NDCG@10: 0.0597
